# Chapter 4 - The Derivative
*Analytic Geometry and Calculus with Python* (4th edition), S. E. Moore.

Run each cell with **Shift + Enter**. All the libraries used (NumPy, SymPy, Matplotlib, SciPy) are pre-installed on Google Colab.

## Listing 4.1: Secant lines turning into the tangent line

In [ ]:
# Listing 4.1: Secant lines turning into the tangent line
import numpy as np
import matplotlib.pyplot as plt

f = lambda x: x**2
a = 1.0
x = np.linspace(-0.5, 3, 200)

plt.figure(figsize=(5.5, 4))
plt.plot(x, f(x), "k", lw=2, label="$y = x^2$")
for h, c in zip([1.5, 1.0, 0.5], ["C1", "C2", "C4"]):
    m = (f(a + h) - f(a)) / h                  # gradient of the secant
    plt.plot(x, f(a) + m*(x - a), c, lw=1, label=f"secant, h = {h}: m = {m:.2f}")
    plt.plot(a + h, f(a + h), "o", color=c)
plt.plot(x, f(a) + 2*(x - a), "C3--", lw=2, label="tangent: m = 2")
plt.plot(a, f(a), "ko")
plt.ylim(-1, 9); plt.grid(alpha=0.3); plt.legend(fontsize=8)
plt.show()

for h in [1, 0.1, 0.01, 0.001]:
    print(f"h = {h:<6} secant gradient = {(f(a + h) - f(a))/h:.4f}")


## Listing 4.2: Differentiation from first principles with SymPy

In [ ]:
# Listing 4.2: Differentiation from first principles with SymPy
import sympy as sp

x, h = sp.symbols("x h")

def derivative_from_definition(f):
    return sp.simplify(sp.limit((f.subs(x, x + h) - f) / h, h, 0))

for f in [4*x - 2, 1/x**2, sp.sqrt(x), x**3, sp.sin(x), 4*x/(x + 1)]:
    print(f"f(x) = {str(f):<12}  f'(x) = {derivative_from_definition(f)}")

# Not differentiable: |x| at 0. The one-sided limits of the difference quotient differ.
q = sp.Abs(h) / h
print("|x| at 0: left limit =", sp.limit(q, h, 0, "-"), ", right limit =", sp.limit(q, h, 0, "+"))


## Listing 4.3: The rules of differentiation, checked with sp.diff

In [ ]:
# Listing 4.3: The rules of differentiation, checked with sp.diff
import sympy as sp

x = sp.symbols("x")
examples = {
    "x^5":                       x**5,
    "1/cbrt(x)":                 1/sp.cbrt(x),
    "x^7 + sqrt(x)":             x**7 + sp.sqrt(x),
    "3 sqrt(x) + 5x^2":          3*sp.sqrt(x) + 5*x**2,
    "(x^2+x-1)(2x+4)":           (x**2 + x - 1)*(2*x + 4),
    "(x^4+2x-1)/(3x^2+5)":       (x**4 + 2*x - 1)/(3*x**2 + 5),
    "x sin x - cos x":           x*sp.sin(x) - sp.cos(x),
    "(x^2+2)/(1+sin x)":         (x**2 + 2)/(1 + sp.sin(x)),
    "tan x/(1 + 2 sec x)":       sp.tan(x)/(1 + 2*sp.sec(x)),
}
for name, f in examples.items():
    d = sp.diff(f, x)
    print(f"d/dx[{name}] = {sp.simplify(d)}")


## Listing 4.4: The chain rule - symbolic examples and a fuel-cost calculation

In [ ]:
# Listing 4.4: The chain rule - symbolic examples and a fuel-cost calculation
import sympy as sp

x = sp.symbols("x")
for f in [(x**2 + x)**-5, sp.sin(x)**5, sp.tan(3*x**2 + 1), sp.cos(x**2 + 1)**3]:
    print(f"d/dx[{f}] = {sp.factor(sp.diff(f, x))}")

# Fuel cost of a Bolt driver (illustrative numbers):
#   price  p = 14 GH¢ per litre,  consumption = 0.08 litres per km,  speed = 40 km/h
dC_dF = 14          # GH¢ per litre
dF_dx = 0.08        # litres per km
dx_dt = 40          # km per hour
dC_dt = dC_dF * dF_dx * dx_dt
print(f"Fuel cost rate dC/dt = {dC_dF} x {dF_dx} x {dx_dt} = GH¢ {dC_dt:.2f} per hour")


## Listing 4.5: Implicit differentiation with sp.idiff

In [ ]:
# Listing 4.5: Implicit differentiation with sp.idiff
import sympy as sp

x, y = sp.symbols("x y")

# Worked examples
print("x y^2 + 6y + x = 0   : dy/dx =", sp.simplify(sp.idiff(x*y**2 + 6*y + x, y, x)))
print("cos(xy) + sin x = 1  : dy/dx =", sp.simplify(sp.idiff(sp.cos(x*y) + sp.sin(x) - 1, y, x)))

# The circle x^2 + y^2 = 25: slope of the tangent at (3, 4)
dydx = sp.idiff(x**2 + y**2 - 25, y, x)
print("x^2 + y^2 = 25       : dy/dx =", dydx, "; at (3, 4) it is", dydx.subs({x: 3, y: 4}))

# Second derivative of x^2 + y^2 = 1  (compare the worked example y^3 y'' + 1 = 0)
d2 = sp.idiff(x**2 + y**2 - 1, y, x, 2)
print("x^2 + y^2 = 1        : y'' =", sp.simplify(d2), "-> y^3 y'' =", sp.simplify((y**3*d2).subs(x**2, 1 - y**2)))


## Listing 4.6: Derivatives of exponential and logarithmic functions

In [ ]:
# Listing 4.6: Derivatives of exponential and logarithmic functions
import sympy as sp

x = sp.symbols("x", positive=True)
examples = {
    "e^(x^2)":                    sp.exp(x**2),
    "e^x sin 2x + e^(cos x)":     sp.exp(x)*sp.sin(2*x) + sp.exp(sp.cos(x)),
    "ln(sin x)":                  sp.log(sp.sin(x)),
    "sin(x^2+1) ln x + ln(4x^3+3)": sp.sin(x**2 + 1)*sp.log(x) + sp.log(4*x**3 + 3),
    "4^x":                        4**x,
    "6^(sin x)":                  6**sp.sin(x),
    "log_4 x":                    sp.log(x, 4),
    "x^(sin x)":                  x**sp.sin(x),
}
for name, f in examples.items():
    print(f"d/dx[{name}] = {sp.diff(f, x)}")


## Listing 4.7: How fast is Ghana's population growing?

In [ ]:
# Listing 4.7: How fast is Ghana's population growing?
import numpy as np
import matplotlib.pyplot as plt

# Population and Housing Census totals (millions)
P2010, P2021 = 24.658823, 30.832019
r = np.log(P2021 / P2010) / 11          # from P2021 = P2010 * e^(11 r)
print(f"growth rate r = {r:.5f} per year  ({100*r:.2f}% per year)")

P = lambda t: P2010 * np.exp(r*t)       # t = years after 2010
dP = lambda t: r * P(t)                 # derivative of a e^(rt) is r a e^(rt)
for year in [2010, 2021, 2026]:
    t = year - 2010
    print(f"{year}: P = {P(t):6.2f} million,  P'(t) = {1e6*dP(t):,.0f} people per year")

t = np.linspace(0, 20, 100)
plt.figure(figsize=(5.5, 3.4))
plt.plot(2010 + t, P(t), label="model $P(t)=24.66\\,e^{rt}$")
plt.plot([2010, 2021], [P2010, P2021], "ro", label="census")
t0 = 11
plt.plot(2010 + t, P(t0) + dP(t0)*(t - t0), "k--", lw=1, label="tangent at 2021")
plt.xlabel("year"); plt.ylabel("millions"); plt.grid(alpha=0.3); plt.legend()
plt.show()


## Listing 4.8: Position, velocity and acceleration of a car (higher derivatives)

In [ ]:
# Listing 4.8: Position, velocity and acceleration of a car (higher derivatives)
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

t = sp.symbols("t")
s = sp.Rational(6, 5)*t**2 - sp.Rational(1, 25)*t**3     # metres, 0 <= t <= 20 s
v = sp.diff(s, t)
a = sp.diff(s, t, 2)
print("s(t) =", s, "\nv(t) =", v, "\na(t) =", a)
t_max = sp.solve(a, t)[0]
print("maximum speed at t =", t_max, "s:  v =", v.subs(t, t_max), "m/s =",
      float(v.subs(t, t_max))*3.6, "km/h")

T = np.linspace(0, 20, 200)
funcs = [sp.lambdify(t, e) for e in (s, v, a)]
fig, ax = plt.subplots(1, 3, figsize=(9, 2.6))
for axis, f, name in zip(ax, funcs, ["s (m)", "v (m/s)", "a (m/s²)"]):
    axis.plot(T, f(T) * np.ones_like(T)); axis.set_title(name); axis.set_xlabel("t (s)")
    axis.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## Listing 4.9: Numerical differentiation - derivatives from data

In [ ]:
# Listing 4.9: Numerical differentiation - derivatives from data
import numpy as np

# (a) How accurate are difference quotients?  f(x) = sin x at x = 1, exact f'(1) = cos 1
f, x0, exact = np.sin, 1.0, np.cos(1.0)
print(f"{'h':>8} {'forward error':>15} {'central error':>15}")
for h in [0.1, 0.01, 0.001, 0.0001]:
    fwd = (f(x0 + h) - f(x0)) / h
    cen = (f(x0 + h) - f(x0 - h)) / (2*h)
    print(f"{h:>8} {abs(fwd - exact):>15.2e} {abs(cen - exact):>15.2e}")

# (b) Speed from GPS data logged by a ride-hailing app every 10 s (illustrative)
time = np.array([0, 10, 20, 30, 40, 50, 60])            # s
dist = np.array([0, 60, 210, 420, 640, 830, 960])       # m
speed = np.gradient(dist, time)                         # central differences inside
for t, v in zip(time, speed):
    print(f"t = {t:>2} s   speed ~ {v:5.1f} m/s = {3.6*v:5.1f} km/h")
